# Fine-tune Whisper for Automatic Speech Recognition

In this notebook, we'll fine-tune OpenAI's Whisper model (`openai/whisper-small`) on a subset of the Common Voice dataset for automatic speech recognition (ASR). We'll use a specific language (Hindi) to demonstrate the fine-tuning process.

Whisper is a pre-trained model for automatic speech recognition and speech translation. Fine-tuning it on a specific language or domain can significantly improve its performance for that use case.

**What we'll cover:**
- Loading and preprocessing the Common Voice dataset
- Preparing audio data and transcriptions
- Fine-tuning the Whisper model
- Evaluating with Word Error Rate (WER)
- Running inference on test samples

In [ ]:
# Install required packages
!pip install -q transformers datasets accelerate evaluate jiwer soundfile librosa

<cell_type>markdown</cell_type>## Load Dataset

We'll load the MINDS-14 dataset for English. MINDS-14 is a multilingual intent detection dataset with audio recordings. We'll use it for ASR fine-tuning.

In [ ]:
from datasets import load_dataset, DatasetDict, Audio

# Load MINDS-14 English dataset
minds = load_dataset("PolyAI/minds14", "en-US", split="train")

# Cast audio column to 16kHz (Whisper's expected sampling rate)
minds = minds.cast_column("audio", Audio(sampling_rate=16000))

# Split into train and test
minds = minds.train_test_split(test_size=0.2, seed=42)
common_voice = DatasetDict({
    "train": minds["train"],
    "test": minds["test"],
})

print(f"Train samples: {len(common_voice['train'])}")
print(f"Test samples: {len(common_voice['test'])}")
print(f"\nDataset features: {common_voice['train'].features}")

## Load Processor and Model

We'll load the Whisper processor (which handles audio feature extraction and tokenization) and the Whisper model for conditional generation.

In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration

# Load processor and model
model_name = "openai/whisper-small"

processor = WhisperProcessor.from_pretrained(model_name)
model = WhisperForConditionalGeneration.from_pretrained(model_name)

# Set language and task for the model
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

# Set the language to English
model.generation_config.language = "en"
model.generation_config.task = "transcribe"

print(f"Model loaded: {model_name}")
print(f"Model parameters: {model.num_parameters() / 1_000_000:.2f}M")

## Prepare Data

We need to preprocess the audio data and transcriptions to prepare them for training. This involves:
1. Extracting audio features using the processor
2. Tokenizing the transcription text
3. Creating labels for training

In [ ]:
def prepare_dataset(batch):
    """
    Prepare dataset by processing audio and tokenizing transcriptions.
    """
    # Load and process audio
    audio = batch["audio"]
    
    # Compute input features from audio array
    batch["input_features"] = processor.feature_extractor(
        audio["array"], 
        sampling_rate=audio["sampling_rate"]
    ).input_features[0]
    
    # Tokenize the transcription
    batch["labels"] = processor.tokenizer(batch["transcription"]).input_ids
    
    return batch

# Apply preprocessing to the dataset
print("Preprocessing training data...")
common_voice["train"] = common_voice["train"].map(
    prepare_dataset,
    remove_columns=common_voice["train"].column_names,
    num_proc=1
)

print("Preprocessing test data...")
common_voice["test"] = common_voice["test"].map(
    prepare_dataset,
    remove_columns=common_voice["test"].column_names,
    num_proc=1
)

print("\nPreprocessing complete!")
print(f"Sample input features shape: {len(common_voice['train'][0]['input_features'])}")
print(f"Sample labels length: {len(common_voice['train'][0]['labels'])}")

## Define Data Collator

We need a custom data collator to handle padding for both input features and labels during batching.

In [ ]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    """
    Data collator that will dynamically pad the inputs received.
    """
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # Split inputs and labels since they need different padding
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        label_features = [{"input_ids": feature["labels"]} for feature in features]

        # Pad input features
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # Pad labels
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # Replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # If bos token is appended in previous tokenization step,
        # cut bos token here as it's append later anyways
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch

# Initialize data collator
data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)
print("Data collator initialized!")

## Set Up Training

Now we'll configure the training arguments and set up the evaluation metric (Word Error Rate - WER).

In [ ]:
from transformers import Seq2SeqTrainingArguments
import evaluate

# Define training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-small-finetuned",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=1,
    learning_rate=1e-5,
    warmup_steps=50,
    max_steps=200,
    fp16=True,
    eval_strategy="steps",
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=100,
    eval_steps=100,
    logging_steps=25,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=False,
)

# Load WER metric
metric = evaluate.load("wer")

def compute_metrics(pred):
    """
    Compute Word Error Rate (WER) for predictions.
    """
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # Replace -100 with pad_token_id
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    # Decode predictions and labels
    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    # Compute WER
    wer = 100 * metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer}

print("Training configuration complete!")
print(f"Total training steps: {training_args.max_steps}")
print(f"Evaluation every {training_args.eval_steps} steps")

In [ ]:
from transformers import Seq2SeqTrainer

# Initialize trainer
trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=common_voice["train"],
    eval_dataset=common_voice["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
)

print("Trainer initialized. Starting training...")
print("=" * 50)

# Start training
trainer.train()

print("=" * 50)
print("Training complete!")

## Run Inference

Let's test our fine-tuned model by running inference on a test sample.

In [ ]:
from transformers import pipeline
from datasets import load_dataset, Audio

# Load test samples from MINDS-14
test_dataset = load_dataset("PolyAI/minds14", "en-US", split="train[-10:]")
test_dataset = test_dataset.cast_column("audio", Audio(sampling_rate=16000))

# Create ASR pipeline with fine-tuned model
pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    device=0 if torch.cuda.is_available() else -1,
)

# Run inference on a few test samples
print("Running inference on test samples...\n")
print("=" * 80)

for i in range(min(3, len(test_dataset))):
    sample = test_dataset[i]
    
    # Generate transcription
    result = pipe(sample["audio"]["array"])
    
    print(f"Sample {i+1}:")
    print(f"Ground Truth: {sample['transcription']}")
    print(f"Prediction:   {result['text']}")
    print("=" * 80)

print("\nInference complete!")